# Bronze Layer — Raw Data Ingestion

**Project:** Bank Customer Churn Analytics (Medallion Architecture)  
**Notebook:** `01_bronze_ingestion`  
**Author:** Harshkumar Patel  
**Last Updated:** June 29, 2026

---

## Overview

This notebook implements the **Bronze layer** of the Medallion Architecture.

The objective is to ingest the raw customer churn dataset into Delta Lake **without modifying the source data**. Metadata is appended to support lineage and auditability, while the original dataset remains unchanged.

---

## Responsibilities

- Read the raw CSV dataset from the landing zone
- Infer the source schema
- Add ingestion metadata
- Store the dataset as a managed Delta table
- Execute basic data quality validation

> **Bronze Principle:** Preserve the source exactly as received. No cleansing, filtering, or business transformations are performed.

---

## Source

| Property | Value |
|----------|-------|
| File | `bank_churn_dataset.csv` |
| Location | `/Volumes/workspace/default/churn_data/` |
| Records | 80,000 |
| Columns | 26 |

---

## Target

| Property | Value |
|----------|-------|
| Database | `churn_bronze` |
| Table | `raw_bank_customers` |
| Format | Delta Lake |
| Partition | `origin_province` |

---

## Pipeline Flow

```
CSV Landing Zone
        │
        ▼
Read with Spark
        │
        ▼
Append Metadata
        │
        ▼
Write Delta Table
        │
        ▼
Validate Load
```

## Step 1 — Create Medallion Databases

Create the Bronze, Silver, and Gold databases used throughout the analytics pipeline.

| Layer | Purpose |
|--------|---------|
| Bronze | Raw ingested data |
| Silver | Cleaned and enriched data |
| Gold | Business-ready analytics |

In [0]:
# creating 3 databases to represent the medallion architecture
# bronze = raw data, silver = cleaned data, gold = business KPIs
# keeps everything organised and separated by purpose

spark.sql("CREATE DATABASE IF NOT EXISTS churn_bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS churn_silver")
spark.sql("CREATE DATABASE IF NOT EXISTS churn_gold")

# Verify all 3 were created successfully
spark.sql("SHOW DATABASES").show()

## Step 2 — Define Source Path

Store the source file location in a single variable for maintainability.

In [0]:
# storing the file path in a variable so i dont have to 
# repeat the long path again and again in the notebook

FILE_PATH = "/Volumes/workspace/default/churn_data/bank_churn_dataset.csv"

## Step 3 — Load Raw Dataset

Read the CSV file into a Spark DataFrame using automatic schema inference.

### Validation

- Verify record count
- Verify column count
- Preview the ingested dataset

In [0]:


df_raw = (spark.read                                # reading the csv file into spark
    .option("header", "true")
    .option("inferSchema", "true")                  # inferSchema automatically detects and assigns the correct data type for each column
    .csv(FILE_PATH))

# Quick confirmation of what we read
print(f"Rows: {df_raw.count()}")
print(f"Columns: {len(df_raw.columns)}")

# Preview the first 5 rows
df_raw.display()

## Step 4 — Inspect Schema

Review the inferred schema to ensure Spark assigned appropriate data types before persisting the data.

In [0]:
# checking the schema to make sure spark assigned 
# the correct data type to each column
# for example exit should be boolean, balance should 
# be integer and last_active_date should be date


df_raw.printSchema()

## Step 5 — Add Metadata and Write Delta Table

Append lineage metadata before writing the dataset as a managed Delta table.

### Metadata Added

| Column | Description |
|---------|-------------|
| `_ingestion_timestamp` | Load timestamp |
| `_source_file` | Source file path |
| `_pipeline_layer` | Pipeline stage (`bronze`) |

In [0]:
# adding 3 extra columns to keep a log of when, where 
# and what layer the data came from - this is called data lineage

# saving as delta instead of csv because delta has 
# better features like ACID transactions and query performance

from pyspark.sql.functions import current_timestamp, lit, col
# Add metadata columns to the raw DataFrame
df_bronze = (df_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_pipeline_layer", lit("bronze")))
# Write to Delta Lake as a managed table in churn_bronze database
(df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("origin_province")                                 #partitionBy to partiton based on diffrent provinces
    .saveAsTable("churn_bronze.raw_bank_customers"))

print(f"Bronze table written successfully")
print(f"Total rows: {df_bronze.count():,}")

## Step 6 — Data Quality Validation

Validate that the ingestion completed successfully.

### Checks Performed

- Source and target row counts match
- Critical columns contain no null values
- Churn distribution is consistent with the source

In [0]:

# Confirm row count
count = spark.table("churn_bronze.raw_bank_customers").count()
print(f"Rows in Delta table: {count:,}")

# Null check on critical columns
spark.sql("""
    SELECT
        SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END)   AS null_id,
        SUM(CASE WHEN exit IS NULL THEN 1 ELSE 0 END)  AS null_churn,
        SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END)   AS null_age,
        SUM(CASE WHEN balance IS NULL THEN 1 ELSE 0 END) AS null_balance
    FROM churn_bronze.raw_bank_customers
""").display()

spark.sql("""
    SELECT 
        exit AS churned,
        COUNT(*) AS customer_count
    FROM churn_bronze.raw_bank_customers
    GROUP BY exit
""").display()

## Bronze Layer Summary

| Metric | Result |
|--------|--------|
| Records Ingested | 80,000 |
| Columns | 26 |
| Metadata Columns Added | 3 |
| Nulls in Critical Fields | 0 |
| Storage Format | Delta Lake |
| Partition Column | `origin_province` |
| Target Table | `churn_bronze.raw_bank_customers` |

### Status

Bronze ingestion completed successfully.

**Next Notebook:** `02_silver_transformation`